<a href="https://colab.research.google.com/github/GinnaGomez09/proyecto_aplicado_javeriana/blob/main/notebooks/04_tokenizacion_lematizacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# ============================================================
# 1. Clonar repositorio y preparar entorno
# ============================================================

!git clone https://github.com/GinnaGomez09/proyecto_aplicado_javeriana.git
%cd proyecto_aplicado_javeriana

Cloning into 'proyecto_aplicado_javeriana'...
remote: Enumerating objects: 348, done.
remote: Counting objects: 100% (206/206), done.
remote: Compressing objects: 100% (186/186), done.
remote: Total 348 (delta 133), reused 18 (delta 18), pack-reused 142 (from 1)
Receiving objects: 100% (348/348), 3.84 MiB | 4.53 MiB/s, done.
Resolving deltas: 100% (187/187), done.
/content/proyecto_aplicado_javeriana/proyecto_aplicado_javeriana


In [10]:
# ============================================================
# 04_TOKENIZACION_LEMATIZACION.ipynb
# Tokenización y lematización del dataset limpio de recetas
# ============================================================

import pandas as pd
import numpy as np
import spacy
import subprocess
import sys
from pathlib import Path
from tqdm import tqdm

tqdm.pandas()

In [11]:
# ------------------------------------------------------------
# 1. Cargar dataset limpio desde carpeta interim
# ------------------------------------------------------------

input_path = "data/interim/recetas_limpias.csv"

recetas = pd.read_csv(input_path)

print("Dimensiones del dataset limpio:", recetas.shape)
print("Número de recetas:", recetas["receta_uuid"].nunique())

recetas.head()

Dimensiones del dataset limpio: (3777, 11)
Número de recetas: 436


,receta_uuid,receta_titulo,ingrediente_id,ingrediente_linea,cantidad_original,cantidad_conv,unidad,ingrediente_nombre,ingrediente_linea_limpia,ingrediente_nombre_limpio,unidad_limpia
0,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,1,1 taza de harina de arepa blanca o amarilla,1,1.000000,taza,harina de arepa blanca o amarilla,1 taza de harina de arepa blanca o amarilla,harina de arepa blanca o amarilla,taza
1,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,2,1 taza de agua tibia,1,1.000000,taza,agua tibia,1 taza de agua tibia,agua tibia,taza
2,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,3,⅓ taza de queso mozzarella o queso blanco rallado,⅓,0.333333,taza,queso mozzarella o queso blanco rallado,1/3 taza de queso mozzarella o queso blanco ra...,queso mozzarella o queso blanco rallado,taza
3,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,4,2 cucharadas de mantequilla,2,2.000000,cucharadas,mantequilla,2 cucharadas de mantequilla,mantequilla,cucharada
4,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,5,Sal,NaN,NaN,NaN,Sal,sal,sal,NaN


In [12]:
# ------------------------------------------------------------
# 2. Verificar columnas disponibles
# ------------------------------------------------------------

print("Columnas disponibles:")
print(recetas.columns.tolist())

Columnas disponibles:
['receta_uuid', 'receta_titulo', 'ingrediente_id', 'ingrediente_linea', 'cantidad_original', 'cantidad_conv', 'unidad', 'ingrediente_nombre', 'ingrediente_linea_limpia', 'ingrediente_nombre_limpio', 'unidad_limpia']


In [13]:
# ------------------------------------------------------------
# 3. Cargar modelo de spaCy en español
# ------------------------------------------------------------

def instalar_modelo_spacy():
    modelo = "es_core_news_sm"

    try:
        spacy.load(modelo)
        print(f"Modelo {modelo} disponible.")
    except OSError:
        print(f"Descargando modelo {modelo}...")
        subprocess.check_call(
            [sys.executable, "-m", "spacy", "download", modelo]
        )

instalar_modelo_spacy()

nlp = spacy.load("es_core_news_sm")

Modelo es_core_news_sm disponible.


In [14]:
# ------------------------------------------------------------
# 4. Definir columnas base para NLP
# ------------------------------------------------------------

col_linea_limpia = "ingrediente_linea_limpia"
col_ingrediente_limpio = "ingrediente_nombre_limpio"

print("Columna de línea limpia:", col_linea_limpia)
print("Columna de ingrediente limpio:", col_ingrediente_limpio)

Columna de línea limpia: ingrediente_linea_limpia
Columna de ingrediente limpio: ingrediente_nombre_limpio


In [15]:
# ------------------------------------------------------------
# 5. Funciones de tokenización y lematización
# ------------------------------------------------------------

def obtener_tokens(texto):
    if pd.isna(texto):
        return []

    doc = nlp(str(texto))

    tokens = [
        token.text
        for token in doc
        if not token.is_space
    ]

    return tokens


def obtener_lemas(texto):
    if pd.isna(texto):
        return []

    doc = nlp(str(texto))

    lemas = [
        token.lemma_
        for token in doc
        if not token.is_space
    ]

    return lemas


def obtener_tokens_limpios(texto):
    if pd.isna(texto):
        return []

    doc = nlp(str(texto))

    tokens_limpios = [
        token.text
        for token in doc
        if not token.is_space
        and not token.is_punct
    ]

    return tokens_limpios


def obtener_lemas_limpios(texto):
    if pd.isna(texto):
        return []

    doc = nlp(str(texto))

    lemas_limpios = [
        token.lemma_
        for token in doc
        if not token.is_space
        and not token.is_punct
    ]

    return lemas_limpios

In [16]:
# ------------------------------------------------------------
# 6. Aplicar tokenización y lematización
# ------------------------------------------------------------

recetas_nlp = recetas.copy()

recetas_nlp["tokens_linea"] = (
    recetas_nlp[col_linea_limpia]
    .progress_apply(obtener_tokens_limpios)
)

recetas_nlp["lemas_linea"] = (
    recetas_nlp[col_linea_limpia]
    .progress_apply(obtener_lemas_limpios)
)

recetas_nlp["tokens_ingrediente"] = (
    recetas_nlp[col_ingrediente_limpio]
    .progress_apply(obtener_tokens_limpios)
)

recetas_nlp["lemas_ingrediente"] = (
    recetas_nlp[col_ingrediente_limpio]
    .progress_apply(obtener_lemas_limpios)
)

recetas_nlp.head()

100%|██████████| 3777/3777 [00:25<00:00, 150.52it/s]


,receta_uuid,receta_titulo,ingrediente_id,ingrediente_linea,cantidad_original,cantidad_conv,unidad,ingrediente_nombre,ingrediente_linea_limpia,ingrediente_nombre_limpio,unidad_limpia,tokens_linea,lemas_linea,tokens_ingrediente,lemas_ingrediente
0,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,1,1 taza de harina de arepa blanca o amarilla,1,1.000000,taza,harina de arepa blanca o amarilla,1 taza de harina de arepa blanca o amarilla,harina de arepa blanca o amarilla,taza,"[1, taza, de, harina, de, arepa, blanca, o, am...","[1, taza, de, harina, de, arepa, blanco, o, am...","[harina, de, arepa, blanca, o, amarilla]","[harina, de, arepa, blanco, o, amarillo]"
1,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,2,1 taza de agua tibia,1,1.000000,taza,agua tibia,1 taza de agua tibia,agua tibia,taza,"[1, taza, de, agua, tibia]","[1, taza, de, agua, tibio]","[agua, tibia]","[agua, tibio]"
2,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,3,⅓ taza de queso mozzarella o queso blanco rallado,⅓,0.333333,taza,queso mozzarella o queso blanco rallado,1/3 taza de queso mozzarella o queso blanco ra...,queso mozzarella o queso blanco rallado,taza,"[1/3, taza, de, queso, mozzarella, o, queso, b...","[1/3, taza, de, queso, mozzarella, o, queso, b...","[queso, mozzarella, o, queso, blanco, rallado]","[queso, mozzarella, o, queso, blanco, rallado]"
3,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,4,2 cucharadas de mantequilla,2,2.000000,cucharadas,mantequilla,2 cucharadas de mantequilla,mantequilla,cucharada,"[2, cucharadas, de, mantequilla]","[2, cucharada, de, mantequilla]",[mantequilla],[mantequilla]
4,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,5,Sal,NaN,NaN,NaN,Sal,sal,sal,NaN,[sal],[sal],[sal],[sal]


In [17]:
# ------------------------------------------------------------
# 7. Crear versiones textuales lematizadas
# ------------------------------------------------------------

recetas_nlp["ingrediente_linea_lematizada"] = (
    recetas_nlp["lemas_linea"]
    .apply(lambda x: " ".join(x))
)

recetas_nlp["ingrediente_nombre_lematizado"] = (
    recetas_nlp["lemas_ingrediente"]
    .apply(lambda x: " ".join(x))
)

recetas_nlp[
    [
        "ingrediente_linea",
        "ingrediente_linea_limpia",
        "ingrediente_linea_lematizada",
        "ingrediente_nombre",
        "ingrediente_nombre_limpio",
        "ingrediente_nombre_lematizado"
    ]
].head(20)

,ingrediente_linea,ingrediente_linea_limpia,ingrediente_linea_lematizada,ingrediente_nombre,ingrediente_nombre_limpio,ingrediente_nombre_lematizado
0,1 taza de harina de arepa blanca o amarilla,1 taza de harina de arepa blanca o amarilla,1 taza de harina de arepa blanco o amarillo,harina de arepa blanca o amarilla,harina de arepa blanca o amarilla,harina de arepa blanco o amarillo
1,1 taza de agua tibia,1 taza de agua tibia,1 taza de agua tibio,agua tibia,agua tibia,agua tibio
2,⅓ taza de queso mozzarella o queso blanco rallado,1/3 taza de queso mozzarella o queso blanco ra...,1/3 taza de queso mozzarella o queso blanco ra...,queso mozzarella o queso blanco rallado,queso mozzarella o queso blanco rallado,queso mozzarella o queso blanco rallado
3,2 cucharadas de mantequilla,2 cucharadas de mantequilla,2 cucharada de mantequilla,mantequilla,mantequilla,mantequilla
4,Sal,sal,sal,Sal,sal,sal
5,8 muslos de pollo sin la piel,8 muslos de pollo sin la piel,8 muslo de pollo sin el piel,muslos de pollo sin la piel,muslos de pollo sin la piel,muslo de pollo sin el piel
6,1 cucharada de aceite vegetal,1 cucharada de aceite vegetal,1 cucharada de aceite vegetal,aceite vegetal,aceite vegetal,aceite vegetal
7,½ taza de cebolla picada,1/2 taza de cebolla picada,1/2 taza de cebolla picado,cebolla picada,cebolla picada,cebollo picado
8,½ de taza de pimientón rojo picado,1/2 de taza de pimienton rojo picado,1/2 de taza de pimienton rojo picado,de taza de pimientón rojo picado,de taza de pimienton rojo picado,de taza de pimienton rojo picado
9,1 diente de ajo picado,1 diente de ajo picado,1 diente de ajo picado,ajo picado,ajo picado,agir picado


In [18]:
# ------------------------------------------------------------
# 8. Validación rápida de resultados NLP
# ------------------------------------------------------------

muestra_nlp = recetas_nlp[
    [
        "ingrediente_linea_limpia",
        "tokens_linea",
        "lemas_linea",
        "ingrediente_nombre_limpio",
        "tokens_ingrediente",
        "lemas_ingrediente"
    ]
].sample(15, random_state=42)

display(muestra_nlp)

,ingrediente_linea_limpia,tokens_linea,lemas_linea,ingrediente_nombre_limpio,tokens_ingrediente,lemas_ingrediente
3530,1 lata de leche condensada,"[1, lata, de, leche, condensada]","[1, lata, de, leche, condensado]",leche condensada,"[leche, condensada]","[leche, condensado]"
999,2 cucharadas de aceite vegetal,"[2, cucharadas, de, aceite, vegetal]","[2, cucharada, de, aceite, vegetal]",aceite vegetal,"[aceite, vegetal]","[aceite, vegetal]"
3023,1 taza de queso mozzarella cortado en cubitos,"[1, taza, de, queso, mozzarella, cortado, en, ...","[1, taza, de, queso, mozzarella, cortado, en, ...",queso mozzarella cortado en cubitos,"[queso, mozzarella, cortado, en, cubitos]","[queso, mozzarella, cortado, en, cubito]"
1550,2 hojas de laurel,"[2, hojas, de, laurel]","[2, hoja, de, laurel]",laurel,[laurel],[laurel]
2428,3 tazas de fresas frescas lavadas y cortadas p...,"[3, tazas, de, fresas, frescas, lavadas, y, co...","[3, taza, de, fresa, fresco, lavado, y, cortad...",fresas frescas lavadas y cortadas por la mitad,"[fresas, frescas, lavadas, y, cortadas, por, l...","[fresa, fresco, lavado, y, cortado, por, el, m..."
3732,2 cucharadas de vinagre blanco,"[2, cucharadas, de, vinagre, blanco]","[2, cucharada, de, vinagre, blanco]",vinagre blanco,"[vinagre, blanco]","[vinagrar, blanco]"
2955,1 taza de fresas cortadas en cubitos,"[1, taza, de, fresas, cortadas, en, cubitos]","[1, taza, de, fresa, cortado, en, cubito]",fresas cortadas en cubitos,"[fresas, cortadas, en, cubitos]","[fresa, cortado, en, cubito]"
3646,1 cucharada de extracto de vainilla,"[1, cucharada, de, extracto, de, vainilla]","[1, cucharada, de, extracto, de, vainilla]",extracto de vainilla,"[extracto, de, vainilla]","[extracto, de, vainilla]"
465,1 cucharada de aceite,"[1, cucharada, de, aceite]","[1, cucharada, de, aceite]",aceite,[aceite],[aceite]
1123,2 cucharadas de leche,"[2, cucharadas, de, leche]","[2, cucharada, de, leche]",leche,[leche],[leche]


In [19]:
# ------------------------------------------------------------
# 9. Revisar frecuencia de lemas de ingredientes
# ------------------------------------------------------------

from collections import Counter

todos_los_lemas = []

for lista_lemas in recetas_nlp["lemas_ingrediente"]:
    todos_los_lemas.extend(lista_lemas)

frecuencia_lemas = Counter(todos_los_lemas)

frecuencia_lemas_df = (
    pd.DataFrame(
        frecuencia_lemas.items(),
        columns=["lema", "frecuencia"]
    )
    .sort_values("frecuencia", ascending=False)
)

display(frecuencia_lemas_df.head(30))

,lema,frecuencia
1,de,1304
31,en,420
40,y,408
21,picado,389
12,sal,328
41,cortado,302
4,o,289
121,para,267
46,molido,196
34,fresco,192


In [20]:
# ------------------------------------------------------------
# 10. Guardar dataset tokenizado y lematizado
# ------------------------------------------------------------

Path("data/interim").mkdir(parents=True, exist_ok=True)

output_path = "data/interim/recetas_tokenizadas_lematizadas.csv"

recetas_nlp.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset tokenizado y lematizado guardado en:")
print(output_path)

print("\nDimensiones finales:", recetas_nlp.shape)
print("Número de recetas:", recetas_nlp["receta_uuid"].nunique())

Dataset tokenizado y lematizado guardado en:
data/interim/recetas_tokenizadas_lematizadas.csv

Dimensiones finales: (3777, 17)
Número de recetas: 436


In [ ]:
from google.colab import files

files.download("data/interim/recetas_tokenizadas_lematizadas.csv")

# Tokenización y lematización

En esta etapa se aplica procesamiento lingüístico al dataset limpio de recetas con el fin de preparar el texto para la extracción de entidades culinarias.

La tokenización permite dividir cada línea de ingrediente en unidades léxicas individuales, mientras que la lematización reduce las palabras a su forma base. Esto ayuda a disminuir la variabilidad morfológica presente en los ingredientes, especialmente en casos como plurales, formas femeninas o variaciones gramaticales.

El procesamiento se realiza sobre las columnas limpias generadas en la etapa anterior:

- `ingrediente_linea_limpia`
- `ingrediente_nombre_limpio`

Como resultado, se generan nuevas columnas con tokens y lemas que servirán como insumo para la etapa posterior de NER culinario, donde se extraerán entidades como cantidad, unidad, ingrediente y características de preparación.